# TrafficLens: Data Lake Architecture

This notebook visualizes the data flow from raw video to Business Aggregates using the Medallion Architecture (Bronze, Silver, Gold).

## 1. Flow Diagram

```mermaid
graph TD
    V("VIOFO A229 Pro (.mp4)")

    subgraph Bronze
        CSV("telemetry/*.csv")
        IMG("frames/*.jpg")
    end

    subgraph Silver
        P1("silver/telemetry/*.parquet")
    end

    subgraph Gold
        P2("dim_trips.parquet")
        P3("ml_training_catalog.parquet")
    end

    V -->|OcrVideoReader| CSV
    V -->|OcrVideoReader| IMG
    
    CSV -->|dbt| P1
    
    P1 -->|dbt| P2
    P1 -->|dbt| P3
```


---
## 2. Layer Details

### 🥉 Bronze — Raw Ingestion
- **Source**: VIOFO A229 Pro `.mp4` dashcam videos.
- **Process**: OCR extraction via `OcrVideoReader` (OpenCV + Tesseract).
- **Output**:
  - `datalake/bronze/telemetry/{video}.csv` — frame-by-frame telemetry.
  - `datalake/bronze/frames/{video}/*.jpg` — extracted frames at 1 FPS.

### 🥈 Silver — Cleaned & Typed
- **Process**: dbt + DuckDB (`stg_telemetry.sql`).
- **Output**: `datalake/silver/telemetry/partition_date=YYYY-MM-DD/*.parquet`
  - Timestamps → `DATETIME`, coordinates → `DOUBLE`, nulls removed.

### 🥇 Gold — Business Aggregates
- **`dim_trips.parquet`**: Trip summary (`trip_id`, `start_time`, `duration`, `avg_speed`, `distance_km`).
- **`ml_training_catalog.parquet`**: High-quality frames for ML training (`speed > 5 km/h`).


In [ ]:
# Example Data Interaction: Inspecting the Gold Layer
import duckdb
import pandas as pd

# Connect to DuckDB instance using our physical Path
conn = duckdb.connect(':memory:')

try:
    # Load business aggregate trips dataset
    df = conn.execute("SELECT * FROM '../datalake/gold/dim_trips.parquet' LIMIT 5").df()
    display(df)
except duckdb.BinderException:
    print("Run the ingestion and dbt pipeline first to populate the Gold layer!")